# DIY Copilot
## Lesson: Function Calling

A coding assistant does much more than generating text: it reads files, searches the code base, runs tests, looks up tickets. All of this is built on one mechanism: **function calling** (also called *tool calling*).

The important thing to understand: **the model never executes anything.** It can only *ask* for a function to be called – by returning the name of the function and the arguments. The program around the model – the **harness** – decides whether to run it, runs it, and sends the result back. GitHub Copilot, Cursor or Claude Code are such harnesses around a model; in this lesson, our notebook code is the harness. We look at every step of this process under the hood.

Our assistant gets one tool: `search_knowledge_base(query)` – the retriever over the company knowledge base from day 1. Instead of retrieving context for *every* question (like in the RAG lessons), the model now decides itself **when** and **what** to search.

During this lesson you will learn:
* how a function is described for the model (JSON schema)
* what is actually sent to and returned by the model provider
* how to control whether the model calls a tool (`tool_choice`)
* how tool results are sent back to the model (message flow)
* how to build the **tool loop** – the core of every coding assistant and agent
* how to handle errors and keep a human in the loop for tools with side effects

### Set up the environment

In [1]:
# >>> COLAB SETUP <<<
# Placeholder: clone the workshop repository into the Colab workspace and make `lib` importable.
# When running locally, the following line is sufficient.
import sys
sys.path.append('../../../')

In [2]:
import json

import litellm
from chromadb import EphemeralClient
from litellm import completion
from litellm.integrations.custom_logger import CustomLogger

from lib.constants import KNOWLEDGE_BASE_PATH
from lib.environment import load_environment
from lib.evaluation import do_top_k_fetching
from lib.rag import do_embed, do_ingestion, list_knowledge_base_files
from lib.retry import call_with_retry

default_models = load_environment()
generation_model = default_models["generation_model"]
embedding_model = default_models["embedding_model"]
print(f"Generation model: {generation_model}")
print(f"Embedding model:  {embedding_model}")

SYSTEM_PROMPT = (
    "You are a helpful coding assistant for the developers of our company. "
    "Use the available tools when a question depends on internal rules or conventions of the company. "
    "If a question has several parts, search separately for each part. "
    "Base your answer only on the search results and cite the sources."
)

Generation model: gemini/gemini-3.1-flash-lite
Embedding model:  gemini/gemini-embedding-001


### The function we want to give to the model
First we build the knowledge base index (like in the RAG lessons) and define the function. It is a completely normal Python function – you can call it yourself.

In [3]:
# Build the index of the knowledge base (one embedding request per chunk).
# Outdated documents (archive/) are not indexed - remember the RAG lessons: they sound right, but they are wrong.
DOCUMENTS = [f for f in list_knowledge_base_files(KNOWLEDGE_BASE_PATH) if "archive/" not in f]

chromadb_client = EphemeralClient()
do_ingestion(
    file_names=DOCUMENTS,
    data_path=KNOWLEDGE_BASE_PATH,
    embedding_model=embedding_model,
    chromadb_client=chromadb_client,
    add_file_header=True,
)

In [4]:
def search_knowledge_base(query: str) -> str:
    """Searches the company's internal knowledge base and returns the most relevant sections.

    The knowledge base contains business policies (e.g. meeting room booking in Berlin and
    Munich, desk sharing, parking), engineering guidelines (Python coding, API design,
    testing, git and code review) and security requirements (secrets, logging, data
    protection, use of AI assistants). Only current documents are searched. Use it whenever
    a question depends on internal rules or conventions of the company.

    Args:
        query: What to search for, phrased like the wording of a policy or guideline.

    Returns:
        The most relevant sections, each with the document and line range it comes from.
    """
    chunks = do_top_k_fetching(do_embed(query, embedding_model), top_k=3, chromadb_client=chromadb_client)
    return "\n\n---\n\n".join(f"Source: {chunk.metadata.origin}\n{chunk.chunk}" for chunk in chunks)


# It's just a Python function:
print(search_knowledge_base("maximum duration of a meeting room booking in Berlin")[:600])

Source: domain/room-booking-policy.md:28-50
# File: domain/room-booking-policy.md
### 2.2 Duration

A booking lasts at least 15 minutes and at most **4 hours**. Rooms cannot be
blocked for a whole day by a single booking. Workshops that need a room for a
full day are booked by the facility team as several consecutive bookings –
please send a request to the facility team at least one week in advance.

### 2.3 Business hours

Rooms can be booked from **Monday to Friday between 07:00 and 20:00** (Berlin
time). A booking must start and end on the same day. There are no bookings on
weekends; for we


### Exercise 01: Describe the function for the model
The model can't see your Python code. It only gets a **description** of the function: its name, what it does, and which parameters it expects – as a [JSON schema](https://json-schema.org/). LiteLLM uses the OpenAI format: each tool is wrapped in `{"type": "function", "function": {...}}`.

The description is the only thing the model knows about the tool. It decides *when* to call the tool and *what to pass* based on these texts – so write them as carefully as a prompt.

Your task is to complete the tool declaration.

In [5]:
# TODO: Complete the declaration of the search_knowledge_base function
search_knowledge_base_tool = {
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": (
            "Searches the company's internal knowledge base (business policies, engineering guidelines, "
            "security requirements) and returns the most relevant sections with their sources. "
            "Use it whenever a question depends on internal rules or conventions of the company."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "What to search for, phrased like the wording of a policy or guideline.",
                },
            },
            "required": ["query"],
        },
    },
}

### Under the hood: what is sent to the model provider?
LiteLLM translates our request into the format of the provider. To see what is actually sent, we register a small **callback** that records the request body before it is sent and the raw response afterwards.

>**Note:** We only print the request *body*. The request URL of the Gemini API contains your API key.

In [6]:
class RequestRecorder(CustomLogger):
    """Records the raw request body and the raw response of the last LLM call."""

    def __init__(self):
        self.request_body = None
        self.raw_response = None

    def log_pre_api_call(self, model, messages, kwargs):
        self.request_body = kwargs.get("additional_args", {}).get("complete_input_dict")

    def log_success_event(self, kwargs, response_obj, start_time, end_time):
        self.raw_response = kwargs.get("original_response")


recorder = RequestRecorder()
litellm.callbacks = [recorder]

In [7]:
# TODO: Ask a question which should make the model call the tool
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the maximum duration of a meeting room booking in Berlin?"},
]
response = call_with_retry(completion, model=generation_model, messages=messages, tools=[search_knowledge_base_tool], temperature=0.0)

# This is what LiteLLM sent to Gemini: our OpenAI-style tool became a "function_declaration"
print(json.dumps(recorder.request_body, indent=1))

{
 "contents": [
  {
   "role": "user",
   "parts": [
    {
     "text": "What is the maximum duration of a meeting room booking in Berlin?"
    }
   ]
  }
 ],
 "system_instruction": {
  "parts": [
   {
    "text": "You are a helpful coding assistant for the developers of our company. Use the available tools when a question depends on internal rules or conventions of the company. If a question has several parts, search separately for each part. Base your answer only on the search results and cite the sources."
   }
  ]
 },
 "tools": [
  {
   "function_declarations": [
    {
     "name": "search_knowledge_base",
     "description": "Searches the company's internal knowledge base (business policies, engineering guidelines, security requirements) and returns the most relevant sections with their sources. Use it whenever a question depends on internal rules or conventions of the company.",
     "parameters": {
      "type": "object",
      "properties": {
       "query": {
        "type": 

In [8]:
# This is what Gemini returned: a "functionCall" part instead of text
raw = json.loads(recorder.raw_response)
for part in raw["candidates"][0]["content"]["parts"]:
    part.pop("thoughtSignature", None)  # internal data of the model, not relevant here
print(json.dumps(raw["candidates"][0], indent=1))

{
 "content": {
  "parts": [
   {
    "functionCall": {
     "name": "search_knowledge_base",
     "args": {
      "query": "maximum duration of a meeting room booking in Berlin"
     },
     "id": "call_17324639160652766175"
    }
   }
  ],
  "role": "model"
 },
 "finishReason": "STOP",
 "index": 0,
 "finishMessage": "Model generated function call(s)."
}


### Exercise 02: Inspect the tool call
LiteLLM converts the provider's answer back into the OpenAI format. Your task is to print the relevant fields of the response:
* `finish_reason` – why did the model stop?
* `content` – is there any text?
* `tool_calls` – for each call: its `id`, the function `name` and the `arguments`

What is the type of `arguments`? Compare it with the raw response of Gemini above.

In [9]:
# TODO: Print the relevant fields of the response
choice = response.choices[0]
print(f"finish_reason: {choice.finish_reason}")
print(f"content:       {choice.message.content}")
for tool_call in choice.message.tool_calls:
    print(f"tool call id:  {tool_call.id[:40]}...")
    print(f"function name: {tool_call.function.name}")
    print(f"arguments:     {tool_call.function.arguments}  ({type(tool_call.function.arguments).__name__})")

finish_reason: tool_calls
content:       None
tool call id:  call_f1306548ac094b40a24de9233c1f__thoug...
function name: search_knowledge_base
arguments:     {"query": "maximum duration of a meeting room booking in Berlin"}  (str)


In [10]:
# The tool description is part of the prompt - it costs tokens with every request
without_tools = call_with_retry(completion, model=generation_model, messages=messages, temperature=0.0, max_tokens=5)
print(f"Prompt tokens without tools: {without_tools.usage.prompt_tokens}")
print(f"Prompt tokens with tools:    {response.usage.prompt_tokens}")

Prompt tokens without tools: 70
Prompt tokens with tools:    174


What did we learn?
* The model did not answer the question. It returned a **request** to call `search_knowledge_base` – and it chose the search query itself.
* Gemini returns the arguments as a JSON object; LiteLLM converts them into a **JSON string** (OpenAI format). The harness has to parse it – and must not trust it blindly.
* Every tool description is sent with every request and costs tokens. Coding assistants with dozens of tools spend thousands of tokens on tool descriptions alone.

### Example: Control the tool usage with `tool_choice`
By default the model decides itself whether to call a tool (`tool_choice="auto"`). You can override this:
* `"none"` – the model must answer with text
* `"required"` – the model must call a tool

Let's send a question that does **not** need the knowledge base with all three settings.

In [11]:
question = "Write a Python function that reverses a string."
for tool_choice in ["auto", "none", "required"]:
    result = call_with_retry(
        completion,
        model=generation_model,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}],
        tools=[search_knowledge_base_tool],
        tool_choice=tool_choice,
        temperature=0.0,
        max_tokens=300,
    )
    message = result.choices[0].message
    calls = [f"{tc.function.name}({tc.function.arguments})" for tc in message.tool_calls or []]
    print(f"{tool_choice:8} -> tool calls: {calls}  text: {(message.content or '')[:60]!r}")

auto     -> tool calls: []  text: 'In Python, the most idiomatic and efficient way to reverse a'
none     -> tool calls: []  text: 'To reverse a string in Python, the most idiomatic and effici'
required -> tool calls: ['search_knowledge_base({"query": "coding standards for Python functions"})']  text: ''


`required` forces a tool call even if it makes no sense. It is useful when the next step of the harness *must* be a tool call – e.g. to force structured data extraction.

### Exercise 03: Execute the function and send the result back
Now we complete the round trip. The conversation is a list of messages, and the tool call becomes part of it:

```
system     You are a helpful coding assistant ...
user       What is the maximum duration of a meeting room booking in Berlin?
assistant  (tool_calls: search_knowledge_base(query="..."), id="call_123")
tool       (tool_call_id="call_123") Source: domain/room-booking-policy.md ...
assistant  A booking in Berlin can last at most 4 hours ...
```

The `tool_call_id` links the result to the call – important when the model requests several calls at once.

Your task is to:
1. Execute the requested function with the parsed arguments.
2. Append the assistant message (with the tool calls) and a `tool` message with the result.
3. Call the model again to get the final answer.

In [12]:
# TODO: Execute the tool call and send the result back to the model
assistant_message = response.choices[0].message
messages.append(assistant_message.model_dump())

for tool_call in assistant_message.tool_calls:
    arguments = json.loads(tool_call.function.arguments)
    result = search_knowledge_base(**arguments)
    messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

final_response = call_with_retry(completion, model=generation_model, messages=messages, tools=[search_knowledge_base_tool], temperature=0.0)
print(final_response.choices[0].message.content)

The maximum duration for a meeting room booking in Berlin is **4 hours**.

According to the [Room Booking Policy](domain/room-booking-policy.md), rooms cannot be blocked for a whole day by a single booking. If you require a room for a full-day workshop, you must send a request to the facility team at least one week in advance so they can arrange it as several consecutive bookings.

**Source:** [domain/room-booking-policy.md](domain/room-booking-policy.md)


In [13]:
# The complete conversation the model saw in the second request
for message in messages:
    content = message.get("content") or ""
    calls = [tc["function"]["name"] for tc in message.get("tool_calls") or []]
    print(f"{message['role']:10} {('tool_calls: ' + str(calls)) if calls else content[:100].replace(chr(10), ' ')}")

system     You are a helpful coding assistant for the developers of our company. Use the available tools when a
user       What is the maximum duration of a meeting room booking in Berlin?
assistant  tool_calls: ['search_knowledge_base']
tool       Source: domain/room-booking-policy.md:28-50 # File: domain/room-booking-policy.md ### 2.2 Duration  


Note that the model is **stateless**: in the second request we send the complete conversation again – including the tool result. Long tool results therefore make every following request more expensive. Keep this in mind for the lesson on context engineering.

### (Optional) Exercise 04: The tool loop
In exercise 03 we did exactly one round trip. But the model may need several tool calls: search, read the result, search again with a better query, and only then answer. The **tool loop** handles this:

1. Call the model with the conversation and the tools.
2. If the answer contains tool calls: execute them, append the results, go back to 1.
3. If the answer is text: done.

This loop is the heart of every coding assistant and every agent. Always limit the number of steps – otherwise a confused model can loop forever (and burn your quota).

The model chooses the function **and** the arguments – so things can go wrong: unknown function names, invalid JSON, wrong parameter names, exceptions in your function. The helper `execute_tool_call` below never crashes: it sends errors back to the model as the tool result, and the model can often fix the problem itself.

>**Note:** The system prompt has a big influence on how the model uses tools. In our tests, without the instruction *"If a question has several parts, search separately for each part. Base your answer only on the search results"*, the model repeated similar searches, mixed up the Berlin and Munich rules and gave a wrong answer. Try it yourself: remove the last two sentences of `SYSTEM_PROMPT` and run the loop again.

In [14]:
def execute_tool_call(tool_call, tools: dict) -> str:
    """Executes a tool call and always returns a string for the model - also in case of errors."""
    name = tool_call.function.name
    if name not in tools:
        return f"Error: unknown tool '{name}'. Available tools: {list(tools)}"
    try:
        arguments = json.loads(tool_call.function.arguments)
    except json.JSONDecodeError:
        return "Error: the arguments are not valid JSON."
    try:
        return str(tools[name](**arguments))
    except Exception as error:
        return f"Error while executing {name}: {type(error).__name__}: {error}"

In [15]:
# TODO: Implement the tool loop
def run_assistant(question: str, tools: dict, tool_declarations: list[dict], max_steps: int = 5) -> str:
    """Answers a question, letting the model call tools as often as it needs (up to max_steps).

    Parameters
    ----------
    question : str
        The question of the developer
    tools : dict
        Mapping of tool names to Python functions
    tool_declarations : list[dict]
        The tool declarations for the model
    max_steps : int
        Maximum number of model calls
    Returns
    -------
    str
        The final answer of the model
    """
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]

    for step in range(1, max_steps + 1):
        response = call_with_retry(completion, model=generation_model, messages=messages, tools=tool_declarations, temperature=0.0)
        message = response.choices[0].message
        messages.append(message.model_dump())

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            print(f"[step {step}] {tool_call.function.name}({tool_call.function.arguments})")
            result = execute_tool_call(tool_call, tools)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "I could not finish the task within the allowed number of steps."

In [16]:
# A question that needs information from two different documents
answer = run_assistant(
    "A developer wants to cancel a meeting room booking in the Berlin office 1 hour before it starts via our API. "
    "Is that allowed, and which HTTP status code should the API return if it is not?",
    tools={"search_knowledge_base": search_knowledge_base},
    tool_declarations=[search_knowledge_base_tool],
)
print(f"\n{answer}")

[step 1] search_knowledge_base({"query": "meeting room booking cancellation policy Berlin office"})
[step 2] search_knowledge_base({"query": "API error code for late meeting room cancellation"})
[step 3] search_knowledge_base({"query": "HTTP status code for late cancellation of meeting room booking Roomie API"})
[step 4] search_knowledge_base({"query": "Roomie API error code for late cancellation"})

No, it is not allowed for a standard user to cancel a meeting room booking 1 hour before it starts.

According to the **Meeting Room Booking Policy**, employees can only cancel their own bookings themselves until **2 hours before the start**. Cancellations made after this time are considered "late cancellations" and are only possible by the facility team.

Regarding the HTTP status code, while the specific API documentation for the "late cancellation" error code was not explicitly found in the provided files, the **API Design Guidelines** specify that error codes should follow the format `

### Example: A human in the loop
Remember indirect prompt injection from the guardrails lesson: the model can be manipulated – and with tools, a manipulated model can **act**. Tools with side effects (creating tickets, writing files, running commands) therefore must not run without a check. A common pattern is to ask a **human for approval** before such a tool runs.

Let's give the assistant a second tool that creates a ticket in the issue tracker (simulated) and extend `execute_tool_call` with an approval step. In a real assistant, the approval is a dialog in the user interface; here a function decides – try changing `deny_everything` to a function that returns `True`.

In [17]:
def create_ticket(title: str, description: str) -> str:
    """Creates a new ticket in the issue tracker of Team Workplace (simulated)."""
    return "Created ticket WORK-431."


create_ticket_tool = {
    "type": "function",
    "function": {
        "name": "create_ticket",
        "description": "Creates a new ticket in the issue tracker of Team Workplace. Returns the id of the new ticket.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string", "description": "A short title of the problem."},
                "description": {"type": "string", "description": "A description of the problem, including how to reproduce it."},
            },
            "required": ["title", "description"],
        },
    },
}

# Tools that change something must be approved by a human before they run
TOOLS_REQUIRING_APPROVAL = {"create_ticket"}


def deny_everything(name: str, arguments: str) -> bool:
    print(f"   Approval requested for {name}({arguments}) -> denied")
    return False


def execute_tool_call(tool_call, tools: dict, approve=deny_everything) -> str:
    """Executes a tool call; tools with side effects need approval first."""
    name = tool_call.function.name
    if name in TOOLS_REQUIRING_APPROVAL and not approve(name, tool_call.function.arguments):
        return "The developer did not approve this action. Do not try again; explain what you wanted to do instead."
    if name not in tools:
        return f"Error: unknown tool '{name}'. Available tools: {list(tools)}"
    try:
        return str(tools[name](**json.loads(tool_call.function.arguments)))
    except Exception as error:
        return f"Error while executing {name}: {type(error).__name__}: {error}"


# run_assistant uses the new execute_tool_call automatically
answer = run_assistant(
    "The availability view of Roomie shows free slots on weekends. Please create a ticket for this bug.",
    tools={"search_knowledge_base": search_knowledge_base, "create_ticket": create_ticket},
    tool_declarations=[search_knowledge_base_tool, create_ticket_tool],
)
print(f"\n{answer}")

[step 1] search_knowledge_base({"query": "Roomie availability view weekend policy"})
[step 2] create_ticket({"description": "The Roomie availability view is currently displaying free slots on weekends. According to the company's meeting room booking policies, meeting rooms should only be available for booking during standard business hours (Monday to Friday). This needs to be restricted to prevent weekend bookings.", "title": "Roomie availability view shows free slots on weekends"})
   Approval requested for create_ticket({"description": "The Roomie availability view is currently displaying free slots on weekends. According to the company's meeting room booking policies, meeting rooms should only be available for booking during standard business hours (Monday to Friday). This needs to be restricted to prevent weekend bookings.", "title": "Roomie availability view shows free slots on weekends"}) -> denied

I have attempted to create a ticket for the issue where Roomie displays free slot

### Learnings
* The model never executes anything – it returns the name of a function and the arguments. The **harness** is in control: it executes the tools, sends the results back and decides when to stop.
* The model only knows the **description** of a tool. Name, description and parameter descriptions are prompts – write them carefully.
* Tool descriptions are sent with every request and cost tokens; tool results become part of the conversation and are sent again with every following request.
* The **tool loop** (call model → execute tools → send results → repeat) is the core of every coding assistant and every agent. Always limit the number of steps.
* Arguments come from the model – validate them, handle errors gracefully, and never let tools with side effects run without a check.
* We wrote the tool description and the loop ourselves. In the next lesson, the same function moves into an **MCP server**, which generates the description automatically and lets any assistant use the tool.